In [7]:
import numpy as np

# Set random seeds for reproducibility
np.random.seed(42)

# Loading and Cleaning Data

In [8]:
import os

DATA_DIR: str = "./data"
RAW_DATA_PATH = os.path.join(DATA_DIR, "_raw_data.txt")
CLEAN_DATA_PATH = os.path.join(DATA_DIR, "clean_data.txt")
REPLACEMENT_MAP: dict[str, str] = {
    "\n": " ",
    "\t": " ",
}
MAX_LINES: int = 5_000

# Load data from a text file
raw_data: list[str] = []
try:
    with open(RAW_DATA_PATH, "r", encoding="utf-8") as f:
        raw_data: list[str] = f.readlines()
    np.random.shuffle(raw_data)  # Shuffle data after loading
    print(f"Loaded {len(raw_data):,d} lines of raw data.")
except FileNotFoundError:
    # Use the cleaned data file as a fallback
    with open(CLEAN_DATA_PATH, "r", encoding="utf-8") as f:
        raw_data: list[str] = f.readlines()
    np.random.shuffle(raw_data)  # Shuffle data after loading
    print(f"Loaded {len(raw_data):,d} lines of cleaned data.")


line_lengths: list[int] = []
for line in raw_data[0:MAX_LINES]:
    # Replace special characters with unique identifiers
    for target, replacement in REPLACEMENT_MAP.items():
        line = line.replace(target, replacement)
    cleaned_data.append(line)
    line_lengths.append(len(line))
print("\nCleaned data.")
print(f"Average line length: {np.mean(line_lengths):.2f} characters.")

# Save cleaned data to a new text file
with open(CLEAN_DATA_PATH, "w", encoding="utf-8") as f:
    for line in cleaned_data:
        f.write(line + "\n")
print(f"\nSaved {len(cleaned_data):,d} lines of cleaned data to {CLEAN_DATA_PATH}.")

Loaded 5,000 lines of cleaned data.

Cleaned data.
Average line length: 120.28 characters.

Saved 10,000 lines of cleaned data to ./data/clean_data.txt.


# Tokenization

In [4]:
from tokenizers import Tokenizer, normalizers, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

UNKNOWN_TOKEN: str = "<UNK>"
ARTIFACT_DIR: str = "./artifacts"
TOKENIZER_PATH: str = os.path.join(ARTIFACT_DIR, "tokenizer.json")
VOCAB_SIZE: int = 10_000

# Initialize a BPE tokenizer
tokenizer: Tokenizer = Tokenizer(BPE(unk_token=UNKNOWN_TOKEN))
tokenizer.normalizer = normalizers.Sequence(
    [
        normalizers.NFKC(),
        normalizers.Lowercase(),  # LLaMA did not use lowercasing, but we will use it to reduce vocab size for training
    ]
)
tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
    [
        pre_tokenizers.WhitespaceSplit(),
        pre_tokenizers.Punctuation(),
        pre_tokenizers.Digits(individual_digits=True),
    ]
)

# Train the tokenizer
trainer: BpeTrainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=[
        UNKNOWN_TOKEN,
    ],
)
tokenizer.train([CLEAN_DATA_PATH], trainer)
tokenizer.save(TOKENIZER_PATH)
print(f"\nTrained and saved tokenizer to {TOKENIZER_PATH}.")





Trained and saved tokenizer to ./artifacts/tokenizer.json.


# Training Data Preparation

In [5]:
TEST_SPLIT_RATIO: float = 0.1
SEQUENCE_LENGTH: int = 512  # Context window size


def create_sequences(tokens: list[int], seq_length: int) -> np.ndarray:
    """Create overlapping sequences for next token prediction."""
    sequences: list[list[int]] = []

    # Use stride to create overlapping windows
    # (stride < seq_length for more data)
    stride: int = seq_length // 2  # 50% overlap

    for i in range(0, len(tokens) - seq_length, stride):
        seq = tokens[i : i + seq_length + 1]  # +1 for the target token
        if len(seq) == seq_length + 1:
            sequences.append(seq)

    return np.array(sequences, dtype=np.int32)


# Tokenize all cleaned data into a continuous sequence
print("Tokenizing data...")
all_tokens: list[int] = []
for line in cleaned_data:
    encoding = tokenizer.encode(line)
    all_tokens.extend(encoding.ids)
print(f"Total tokens: {len(all_tokens):,d}")
print(f"Vocabulary size: {tokenizer.get_vocab_size():,d}")

# Split tokens into train and test
split_idx: int = int(len(all_tokens) * (1 - TEST_SPLIT_RATIO))
train_tokens: list[int] = all_tokens[:split_idx]
test_tokens: list[int] = all_tokens[split_idx:]
print(f"\nTrain tokens: {len(train_tokens):,d}")
print(f"Test tokens: {len(test_tokens):,d}")

# Create sequences for next token prediction
train_sequences: np.ndarray = create_sequences(train_tokens, SEQUENCE_LENGTH)
test_sequences: np.ndarray = create_sequences(test_tokens, SEQUENCE_LENGTH)
print(f"\nTrain sequences: {len(train_sequences):,d}")
print(f"Test sequences: {len(test_sequences):,d}")
print(f"Sequence shape: {train_sequences.shape}")

# Save as numpy arrays for efficient loading during training
TRAIN_SEQUENCES_PATH = os.path.join(DATA_DIR, "train_sequences.npy")
TEST_SEQUENCES_PATH = os.path.join(DATA_DIR, "test_sequences.npy")
np.save(TRAIN_SEQUENCES_PATH, train_sequences)
np.save(TEST_SEQUENCES_PATH, test_sequences)
print(f"\nSaved train sequences to {TRAIN_SEQUENCES_PATH}")
print(f"Saved test sequences to {TEST_SEQUENCES_PATH}")

Tokenizing data...
Total tokens: 139,058
Vocabulary size: 10,000

Train tokens: 125,152
Test tokens: 13,906

Train sequences: 487
Test sequences: 53
Sequence shape: (487, 513)

Saved train sequences to ./data/train_sequences.npy
Saved test sequences to ./data/test_sequences.npy
